# 04 — Haiqu Hardware Deployment: GNN-HVA on Real QPU

Run an HVA circuit with **GNN-predicted parameters** on a real QPU using the
[Haiqu](https://docs.haiqu.ai) middleware stack — state compression + automatic
error mitigation + hardware execution — through the project's `HaiquBackend`.

**End-to-end flow:**
1. Build the TFIM Hamiltonian and exact ground truth (E_exact, gap) per h.
2. Predict θ with the trained MPNN (zero-shot on unseen h-values).
3. Build the HVA circuit (`create_pauli_evolution`).
4. (Optional) Compress the circuit with `haiqu.state_compression`.
5. Estimate QPU cost with a **dry run** (spends no credits).
6. Execute with **Haiqu Error Shield** (`use_mitigation=True`) in observable mode.
7. Reconstruct ⟨H⟩ and report ΔE/gap.

**Canonical Haiqu recommendation:** compress first, then run with mitigation —
the two features are designed to work together.

**Requirements:** `pip install -e .` + `pip install haiqu-sdk`, and a Haiqu API key
(`HAIQU_API_KEY`). For real IBM devices you also need `IBM_KEY` + `IBM_INSTANCE_CRN`.
For credential-free exploration use `device_id="fake_torino"` or `"aer_simulator"`.

## 1. Imports

Everything comes from the existing project modules plus the `HaiquBackend`
wrapper. The Haiqu SDK itself is imported lazily inside the backend, so this
cell works even before `haiqu-sdk` is installed.

In [ ]:
import numpy as np

# Pipeline core (same imports as notebook 01)
from qmbp_simulation import (
    HamiltonianBuilder,
    make_lattice,
    ClassicalSolver,
    HVACircuitBuilder,
)
from qmbp_simulation.predictors import MPNNPredictor, predict_theta
from qmbp_simulation.predictors.mpnn import load_mpnn_checkpoint

# Haiqu integration (wraps the Haiqu cloud stack behind ExecutionBackend)
from qmbp_simulation.execution.hardware.haiqu_backend import HaiquBackend, HaiquConfig

print("✅ Imports OK")

## 2. Configuration

Validated thesis target: `heavy_hex`, N=10, p=1, on `ibm_kingston` (Heron R2).

Start on a **simulator** (`fake_torino`) to validate the flow with no credentials
and no cost, then switch `DEVICE_ID` to a real device when ready.

In [ ]:
# ── Physics / circuit config ─────────────────────────────────────────────
TOPOLOGY = "heavy_hex"
N_QUBITS = 10
P_LAYERS = 1
J = 1.0
H_TEST = [3.5, 3.25]          # unseen h-values for zero-shot deployment

# ── Haiqu execution config ───────────────────────────────────────────────
# fake_torino = credential-free simulator for validating the flow.
# Switch to "ibm_kingston" (and set IBM_KEY / IBM_INSTANCE_CRN) for hardware.
DEVICE_ID = "fake_torino"
USE_MITIGATION = True         # Error Shield (auto multi-layer EM)
USE_COMPRESSION = False       # p=1 N=10 is shallow; enable for p>=2 / large N
SHOTS = 16384

# MPNN checkpoint (produced by the deployment / training pipeline).
# Adjust to your trained model. If missing, section 4 shows how to train one.
MPNN_CHECKPOINT = (
    f"results/hardware/mpnn_checkpoints/mpnn_{TOPOLOGY}_n{N_QUBITS}_p{P_LAYERS}_seed42.pt"
)

cfg = HaiquConfig(
    device_id=DEVICE_ID,
    shots=SHOTS,
    use_mitigation=USE_MITIGATION,
    use_compression=USE_COMPRESSION,
    compression_level="balanced",
    fine_tuning="low",
    # noise_profile auto-selected from device_id (ibm_kingston -> ibm_heron_r2)
    experiment_name="GNN-HVA hardware deployment (notebook 04)",
)
print(f"Device: {cfg.device_id}  |  simulator={cfg.is_simulator()}")
print(f"Compression noise profile: {cfg.resolved_noise_profile()}")
print(f"Mitigation: {cfg.use_mitigation}  |  Compression: {cfg.use_compression}")

## 3. Ground truth: exact E and gap per h

We compute the exact ground-state energy and spectral gap so we can score the
hardware result as ΔE/gap. TFIM convention: `H = -J Σ Z_iZ_j - h Σ X_i`.

In [ ]:
builder = HamiltonianBuilder()
solver = ClassicalSolver()

hamiltonians = {}
e_exact = {}
gap = {}
for h in H_TEST:
    lat_h = make_lattice(TOPOLOGY, N_QUBITS, J=J, h=h)
    H = builder.build(lat_h)
    gt = solver.solve(H, lat_h)
    hamiltonians[h] = H
    e_exact[h] = gt.ground_energy
    gap[h] = gt.gap
    print(f"h={h:.2f}  E_exact={gt.ground_energy:+.4f}  gap={gt.gap:.4f}")

## 4. Predict θ with the GNN (zero-shot)

Load the trained MPNN and predict θ for the unseen test h-values. The predicted
θ vector matches the HVA `ParameterVector` ordering (θ[0]=θ_zz, θ[1]=θ_x for p=1).

> If you don't have a checkpoint yet, generate one with the deployment helper
> `prepare_mpnn_predictions` in
> `scripts/experiment_runners/hardware/run_ibm_deployment.py`, or train via the
> pipeline in notebook 01.

In [ ]:
from pathlib import Path

lattice = make_lattice(TOPOLOGY, N_QUBITS, J=J, h=H_TEST[0])

if Path(MPNN_CHECKPOINT).exists():
    model = load_mpnn_checkpoint(MPNN_CHECKPOINT)
    print(f"Loaded MPNN checkpoint: {MPNN_CHECKPOINT}")
else:
    raise FileNotFoundError(
        f"MPNN checkpoint not found: {MPNN_CHECKPOINT}\n"
        "Generate one via prepare_mpnn_predictions() in run_ibm_deployment.py "
        "or train through notebook 01, then set MPNN_CHECKPOINT above."
    )

theta_per_h = predict_theta(model, lattice, H_TEST)
for h in H_TEST:
    print(f"h={h:.2f}  θ_pred={np.round(theta_per_h[h], 4)}")

## 5. Build the HVA circuit

`create_pauli_evolution` exposes the commuting layer structure to the
transpiler (lower depth). The circuit is **parameterized** — θ is bound by
Haiqu at submission time (or before compression).

In [ ]:
hva = HVACircuitBuilder()
circuit, theta = hva.create_pauli_evolution(N_QUBITS, P_LAYERS, lattice)
print(f"HVA circuit: {circuit.num_qubits} qubits, {circuit.num_parameters} params")
print(f"Parameter order: {list(circuit.parameters)}")
circuit.draw("text", fold=-1)

## 6. Instantiate the Haiqu backend

`HaiquBackend` handles login/init lazily and implements the project's
`ExecutionBackend` contract (`evaluate(circuit, hamiltonian, params) -> energy`).

In [ ]:
backend = HaiquBackend(cfg)
print(f"Backend ready: {backend.name}")

## 7. (Optional) Compress the circuit

`state_compression` needs a **bound** circuit, so we bind θ for one h and
compress. `quality` is Haiqu's fidelity-like approximation metric in [0, 1].
Skipped when `USE_COMPRESSION=False` (p=1 N=10 is already shallow).

In [ ]:
if cfg.use_compression:
    h0 = H_TEST[0]
    compressed, quality = backend.compress_circuit(circuit, theta_per_h[h0])
    print(f"Compression quality (h={h0:.2f}): {quality:.4f}")
    print(f"Original 2Q depth : {circuit.depth(lambda ins: len(ins.qubits) == 2)}")
    print(f"Compressed 2Q depth: {compressed.depth(lambda ins: len(ins.qubits) == 2)}")
else:
    print("Compression disabled (USE_COMPRESSION=False). Skipping.")

## 8. Dry run: estimate QPU cost (no credits spent)

`dry_run=True` returns Haiqu's cost estimate without executing. Use this to
size the run before committing QPU budget.

In [ ]:
h0 = H_TEST[0]
try:
    est = backend.estimate_cost(circuit, hamiltonians[h0], theta_per_h[h0])
    print(f"Estimated QPU cost for h={h0:.2f}: {est}")
except Exception as exc:
    print(f"Cost estimation unavailable ({type(exc).__name__}: {exc}).")
    print("On simulators the dry-run cost may be empty; proceed to execution.")

## 9. Execute on QPU with Error Shield

For each test h, `backend.evaluate` runs the circuit in **observable mode**
(no terminal measurements + Hamiltonian Pauli terms as observables) with
`use_mitigation=True`, then reconstructs ⟨H⟩. We score each point as ΔE/gap
(< 5% is the project's PASS criterion).

In [ ]:
# evaluate_full captures EVERYTHING Haiqu exposes (raw EVs, uncertainty,
# qpu_cost, timing, mitigation flags, compression quality) PLUS derived
# physics (|ΔE|, ΔE/gap, fidelity). All of it is retained in backend.records.
results = []
for h in H_TEST:
    rec = backend.evaluate_full(
        circuit,
        hamiltonians[h],
        theta_per_h[h],
        h=h,
        e_exact=e_exact[h],
        gap=gap[h],
        # exact_state=...,  # optional: pass gt.ground_state for fidelity (N<=~20)
    )
    results.append(rec)
    tag = "✅ PASS" if rec["pass_5pct"] else "❌ FAIL"
    print(
        f"h={h:.2f}  E={rec['energy']:+.4f}  E_exact={rec['e_exact']:+.4f}  "
        f"ΔE/gap={rec['de_gap']:.2%}  unc={rec['uncertainty']}  {tag}"
    )

## 10. Summary

In [ ]:
import pandas as pd

cols = ["h", "energy", "e_exact", "gap", "abs_delta_e", "de_gap", "fidelity", "uncertainty", "pass_5pct"]
df = pd.DataFrame([{c: r.get(c) for c in cols} for r in results])
n_pass = int(df["pass_5pct"].sum())
print(df.to_string(index=False))
print(f"\nPASS rate: {n_pass}/{len(df)}  ({n_pass / len(df):.0%})")
print(f"Mean ΔE/gap: {df['de_gap'].mean():.2%}")

## 11. Save ALL collected data

Persist every Haiqu operation record (compression, dry-run cost, run,
evaluate_full) with all metadata, error metrics, mitigation flags, timing,
QPU cost, and derived physics (|ΔE|, ΔE/gap, fidelity) to a single JSON file.
Credentials are never written.

In [ ]:
from datetime import datetime, timezone

ts = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
out_path = f"results/haiqu/haiqu_{TOPOLOGY}_n{N_QUBITS}_p{P_LAYERS}_{cfg.device_id}_{ts}.json"

backend.save_collected_data(
    out_path,
    extra={
        "topology": TOPOLOGY,
        "n_qubits": N_QUBITS,
        "p_layers": P_LAYERS,
        "J": J,
        "h_test": H_TEST,
        "mpnn_checkpoint": MPNN_CHECKPOINT,
    },
)
print(f"Saved {len(backend.records)} records → {out_path}")

## Notes & next steps

- **Go to hardware:** set `DEVICE_ID = "ibm_kingston"` and export `IBM_KEY` +
  `IBM_INSTANCE_CRN` (or call `haiqu.save_ibm_credentials(...)` once). The
  compression `noise_profile` auto-switches to `ibm_heron_r2`.
- **Deeper circuits (p>=2 / large N):** set `USE_COMPRESSION=True`. Compression
  reduces 2Q depth before mitigation cleans up the rest — the recommended
  combination for production QPU runs.
- **Error bars:** `backend.last_uncertainty` exposes the statistical
  uncertainty Haiqu reports for the observable run.
- **Cost control:** always run section 8 (dry run) before section 9 on real
  hardware.